# Comparing Models for Daily Rat Sightings in New York City

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime

from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_squared_error, mean_absolute_percentage_error
from sklearn.linear_model import LinearRegression

from prophet import Prophet
from pandas.tseries.holiday import USFederalHolidayCalendar
from prophet.diagnostics import cross_validation, performance_metrics
from prophet.plot import plot_cross_validation_metric
from prophet.plot import add_changepoints_to_plot
import itertools

import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.simplefilter('ignore', ConvergenceWarning)

## Importing the Data

In [2]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

In [3]:
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])

In [4]:
# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

In [5]:
# Group and make count column
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')

In [6]:
# This is 2251 which equals the number of days from 2020-01-01 to 2026-02-28
len(rs)

2251

## Baseline Seasonal Average Model

In [7]:
years_back_use = 4
day_window_use = 4

In [8]:
def seasonal_average_forecast(data, target_dates, years_back=years_back_use, day_window=day_window_use):
    df = data.copy()
    df["created_date"] = pd.to_datetime(df["created_date"])
    df["doy"] = df["created_date"].dt.dayofyear
    df["year"] = df["created_date"].dt.year

    forecasts = []
    for target_date in target_dates:
        target_doy = target_date.dayofyear
        target_year = target_date.year
        mask = ((df["year"] >= target_year - years_back) & (df["year"] < target_year) & (np.abs(df["doy"] - target_doy) <= day_window))
        forecasts.append(df.loc[mask, "count"].mean())
    return pd.Series(forecasts, index=target_dates)

In [9]:
results = []
rs["created_date"] = pd.to_datetime(rs["created_date"])



for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    
    train = rs.iloc[train_index].copy()
    test = rs.iloc[test_index].copy()
    
    # Target dates = the dates we want to forecast. There are 14 days.
    target_dates = test["created_date"]
    
    # Seasonal forecast using only the training data (we will go back 5 years and take the average and use a day_window of 5 as well.)
    y_pred = seasonal_average_forecast(data=train, target_dates=target_dates, years_back=years_back_use,day_window=day_window_use)

    # We take the true values.
    y_true = test["count"].values
    
    # Compute the metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append the results of the metrics to the table as well as the fold number.
    results.append({"fold": i, "rmse": rmse, "mape": mape})

# Convert the data to a table for readability.
baseline_results_df = pd.DataFrame(results)

# We also include a new row which consists of the average RMSE and MAPE over each fold.
baseline_results_df.loc["mean"] = ["mean", baseline_results_df["rmse"].mean(), baseline_results_df["mape"].mean()]

baseline_results_df

,fold,rmse,mape
0,0,15.307003,0.285977
1,1,11.914386,0.186347
2,2,14.988228,0.250230
3,3,20.818662,0.348457
4,4,17.563835,0.256820
5,5,23.244116,0.361006
6,6,19.488233,0.266129
7,7,21.222927,0.284166
8,8,20.494910,0.301036
9,9,21.193296,0.277230


## Year Ago Rolling 4 Week Average 

In [10]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

## Just saving a copy for later
rs_saved = rs.copy()

In [11]:
# Tired of writing np.sqrt or typing a long name.
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

results = []

for fold, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    # Calculate the 4-week rolling average for the training data
    train_sorted = train.sort_values('ds') # making sure to sort it by date
    train_sorted['rolling_4w'] = train_sorted['y'].rolling(window=4, min_periods=1).mean()

    # This part of the code makes the predictions. We use the 'rolling_4w' column of the training set.
    y_pred = []
    y_true = test['y'].values

    for idx, row in test.iterrows():
        # Predict using the latest rolling average from the train data
        prediction = train_sorted['rolling_4w'].iloc[-1]  # Last value in the train rolling avg
        y_pred.append(prediction)
        
    # Calculate RMSE and MAPE for this fold
    fold_rmse = rmse(y_true, y_pred)
    fold_mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': fold, 'rmse': fold_rmse, 'mape': fold_mape})

rolling4w_results_df = pd.DataFrame(results)

overall_rmse = rolling4w_results_df['rmse'].mean()
overall_mape = rolling4w_results_df['mape'].mean()
rolling4w_results_df.loc['mean'] = ['mean', overall_rmse, overall_mape]

In [12]:
rolling4w_results_df

,fold,rmse,mape
0,0,14.323744,0.239272
1,1,12.719150,0.196755
2,2,12.398157,0.196394
3,3,19.240721,0.260374
4,4,15.923646,0.220183
5,5,17.378918,0.256428
6,6,24.564601,0.247419
7,7,20.989793,0.255484
8,8,19.858428,0.253080
9,9,18.798509,0.187194


## Prophet Model

In [13]:
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1})

holidays = federal_holidays

In [14]:
# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

prophet_results_df = pd.DataFrame(results)

01:20:28 - cmdstanpy - INFO - Chain [1] start processing
01:20:28 - cmdstanpy - INFO - Chain [1] done processing
01:20:29 - cmdstanpy - INFO - Chain [1] start processing
01:20:29 - cmdstanpy - INFO - Chain [1] done processing
01:20:30 - cmdstanpy - INFO - Chain [1] start processing
01:20:30 - cmdstanpy - INFO - Chain [1] done processing
01:20:30 - cmdstanpy - INFO - Chain [1] start processing
01:20:31 - cmdstanpy - INFO - Chain [1] done processing
01:20:31 - cmdstanpy - INFO - Chain [1] start processing
01:20:32 - cmdstanpy - INFO - Chain [1] done processing
01:20:32 - cmdstanpy - INFO - Chain [1] start processing
01:20:32 - cmdstanpy - INFO - Chain [1] done processing
01:20:33 - cmdstanpy - INFO - Chain [1] start processing
01:20:33 - cmdstanpy - INFO - Chain [1] done processing
01:20:34 - cmdstanpy - INFO - Chain [1] start processing
01:20:34 - cmdstanpy - INFO - Chain [1] done processing
01:20:35 - cmdstanpy - INFO - Chain [1] start processing
01:20:35 - cmdstanpy - INFO - Chain [1]

In [15]:
prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]

In [16]:
prophet_results_df

,fold,rmse,mape
0,0,12.931175,0.246172
1,1,9.286185,0.142636
2,2,7.983666,0.130110
3,3,9.817654,0.143238
4,4,10.903003,0.138791
5,5,15.534582,0.217436
6,6,11.539387,0.145670
7,7,11.299325,0.135217
8,8,10.559222,0.118595
9,9,9.146496,0.113947


## SARIMAX Model

The reason why the SARIMA / SARIMAX model does not perform as well as we'd like is discussed here: https://stats.stackexchange.com/questions/613677/using-sarimax-for-daily-data-with-yearly-seasonal-pattern. An excellent read for more details can be found here: https://robjhyndman.com/hyndsight/longseasonality/. For these reasons, instead of using SARIMA's included seasonality features, we instead add Fourier terms as exogeneous variables.

In [17]:
pip install pmdarima

Note: you may need to restart the kernel to use updated packages.


In [18]:
from pmdarima import auto_arima

In [19]:
def fourier_terms(df, periods, n_terms):
    t = np.arange(1, len(df) + 1)
    fourier_df = pd.DataFrame()
    
    for period in periods:
        for i in range(1, n_terms + 1):
            fourier_df[f'{period}sin_{i}'] = np.sin(2 * np.pi * i * t / period)
            fourier_df[f'{period}cos_{i}'] = np.cos(2 * np.pi * i * t / period)
    
    return fourier_df

fourier_train = fourier_terms(rs, [365,30], 10)
exog = fourier_train

In [20]:
exog

,365sin_1,365cos_1,365sin_2,365cos_2,365sin_3,365cos_3,365sin_4,365cos_4,365sin_5,365cos_5,...,30sin_6,30cos_6,30sin_7,30cos_7,30sin_8,30cos_8,30sin_9,30cos_9,30sin_10,30cos_10
0,0.017213,0.999852,0.034422,0.999407,0.051620,0.998667,0.068802,0.997630,0.085965,0.996298,...,9.510565e-01,0.309017,9.945219e-01,0.104528,9.945219e-01,-0.104528,9.510565e-01,-0.309017,8.660254e-01,-0.5
1,0.034422,0.999407,0.068802,0.997630,0.103102,0.994671,0.137279,0.990532,0.171293,0.985220,...,5.877853e-01,-0.809017,2.079117e-01,-0.978148,-2.079117e-01,-0.978148,-5.877853e-01,-0.809017,-8.660254e-01,-0.5
2,0.051620,0.998667,0.103102,0.994671,0.154309,0.988023,0.205104,0.978740,0.255353,0.966848,...,-5.877853e-01,-0.809017,-9.510565e-01,-0.309017,-9.510565e-01,0.309017,-5.877853e-01,0.809017,-1.133108e-15,1.0
3,0.068802,0.997630,0.137279,0.990532,0.205104,0.978740,0.271958,0.962309,0.337523,0.941317,...,-9.510565e-01,0.309017,-4.067366e-01,0.913545,4.067366e-01,0.913545,9.510565e-01,0.309017,8.660254e-01,-0.5
4,0.085965,0.996298,0.171293,0.985220,0.255353,0.966848,0.337523,0.941317,0.417194,0.908818,...,-1.133108e-15,1.000000,8.660254e-01,0.500000,8.660254e-01,-0.500000,3.673940e-16,-1.000000,-8.660254e-01,-0.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2246,0.831171,0.556017,0.924291,-0.381689,0.196673,-0.980469,-0.705584,-0.708627,-0.981306,0.192452,...,5.877853e-01,-0.809017,9.510565e-01,-0.309017,9.510565e-01,0.309017,5.877853e-01,0.809017,-5.816407e-15,1.0
2247,0.840618,0.541628,0.910605,-0.413279,0.145799,-0.989314,-0.752667,-0.658402,-0.961130,0.276097,...,-5.877853e-01,-0.809017,-2.079117e-01,-0.978148,2.079117e-01,-0.978148,5.877853e-01,-0.809017,8.660254e-01,-0.5
2248,0.849817,0.527078,0.895839,-0.444378,0.094537,-0.995521,-0.796183,-0.605056,-0.933837,0.357698,...,-9.510565e-01,0.309017,-9.945219e-01,0.104528,-9.945219e-01,-0.104528,-9.510565e-01,-0.309017,-8.660254e-01,-0.5
2249,0.858764,0.512371,0.880012,-0.474951,0.043022,-0.999074,-0.835925,-0.548843,-0.899631,0.436651,...,-1.528508e-13,1.000000,-1.783259e-13,1.000000,2.509463e-13,1.000000,2.254712e-13,1.000000,-2.547513e-13,1.0


In [22]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Make sure the columns for SARIMA model are renamed.
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

results = []

# Loop through each fold
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]

    exog_train = exog.iloc[train_index]
    exog_test = exog.iloc[test_index]

    orders = (3,1,3)
    seasonal_orders = (0,0,0,7)

    # we can use auto_arima to get optimal (p, d, q) and (P, D, Q, s) parameters for SARIMAX. just need to uncomment the following code.
    # model_auto = auto_arima(train['y'], 
    #                         exog=exog_train,  # exogenous Fourier terms for training data
    #                         seasonal=True, 
    #                         m=7, #  
    #                         trace=True, 
    #                         stepwise=True,  # Stepwise search to speed up
    #                         suppress_warnings=True, 
    #                         maxiter=300,  # Limit the number of iterations
    #                         max_p=3, 
    #                         max_q=3, 
    #                         max_P=2, 
    #                         max_Q=2, 
    #                         d=1,# might want to tune d 
    #                         D=1 # might want to tune D
    #                         )
    # orders = model_auto.order  # (p, d, q)
    # seasonal_orders = model_auto.seasonal_order  # (P, D, Q, s)
    
    # Fit the SARIMAX model with the exogenous features (Fourier terms)
    model_sarimax = SARIMAX(train['y'], 
                            order=orders,  
                            seasonal_order=seasonal_orders,  
                            exog=exog_train,  # Exogenous Fourier terms for training data
                            )
    
    model_fit = model_sarimax.fit(disp=False)
    
    # Predict for the test period. Have to remember to subtract 1 to get the correct index.
    y_pred = model_fit.predict(start=len(train), end=len(train)+len(test)-1, exog=exog_test, dynamic=False)
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

sarima_results_df = pd.DataFrame(results)

In [23]:
sarima_results_df.loc['mean'] = ['mean',  sarima_results_df['rmse'].mean(), sarima_results_df['mape'].mean()]

In [24]:
sarima_results_df

,fold,rmse,mape
0,0,15.484218,0.239384
1,1,10.325060,0.145598
2,2,10.673432,0.181780
3,3,13.541181,0.198057
4,4,13.465575,0.181587
5,5,15.979406,0.231042
6,6,14.095096,0.151393
7,7,17.213794,0.181633
8,8,16.312946,0.194732
9,9,10.247811,0.112140


## Holt-Winters Model

In [25]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing

In [26]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []
for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    holt_winters = ExponentialSmoothing(train['y'], seasonal='add', seasonal_periods=365).fit(optimized=True)
    
    y_pred = holt_winters.forecast(len(test))
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

hw_results_df = pd.DataFrame(results)
hw_results_df.loc['mean'] = ['mean',  hw_results_df['rmse'].mean(), hw_results_df['mape'].mean()]
hw_results_df

,fold,rmse,mape
0,0,17.029947,0.279665
1,1,17.558940,0.255894
2,2,17.312295,0.297496
3,3,22.556060,0.358384
4,4,19.966872,0.268691
5,5,20.725658,0.310186
6,6,22.555124,0.270105
7,7,25.834040,0.311909
8,8,23.343693,0.306820
9,9,21.316698,0.274190


## XGBoost Model

The XGBoost model requires a bit more preparatory work. Our current dataframe rs is quite bare. We will need to add features for use.

In [27]:
import xgboost as xgb
from xgboost import plot_importance

### Adding Features to XGBoost

In [28]:
def create_features(df):
    # create time series features based on time series index.
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    # features to handly cyclic behavior
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    # lags
    target_map = df['y'].to_dict()
    df['lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
    df['lag3'] = (df.index - pd.Timedelta('3 days')).map(target_map)
    df['lag4'] = (df.index - pd.Timedelta('4 days')).map(target_map)
    df['lag5'] = (df.index - pd.Timedelta('5 days')).map(target_map)
    df['lag6'] = (df.index - pd.Timedelta('6 days')).map(target_map)
    df['lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['lag8'] = (df.index - pd.Timedelta('8 days')).map(target_map)
    df['lag9'] = (df.index - pd.Timedelta('9 days')).map(target_map)
    df['lag10'] = (df.index - pd.Timedelta('10 days')).map(target_map)
    df['lag11'] = (df.index - pd.Timedelta('11 days')).map(target_map)
    df['lag12'] = (df.index - pd.Timedelta('12 days')).map(target_map)
    df['lag13'] = (df.index - pd.Timedelta('13 days')).map(target_map)
    df['lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    # lags of various lengths for different levels of seasonality
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    df['ma7'] = df['y'].shift(1).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(1).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(1).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(1).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(1).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(1).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(1).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(1).rolling(window=365).mean()
    
    return df


In the next two code block, we add weather data to the data set. This is not optimized i.e. we just obtain the weather data in Manhattan and hope that it is representative of the average weather over the whole city.

In [29]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd
    
else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [30]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index 
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary columns
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

In [31]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format handling
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD format handling
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

In [32]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    df.index = pd.to_datetime(df.index)
    start_dt = pd.to_datetime(start_date)
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

In [33]:
# This must be run after importing weather data

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['apparent_temperature_min_lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df

In [34]:
rs = rs.set_index('ds')
rs.index = pd.to_datetime(rs.index)

In [35]:
rs = create_features(rs)
rs = add_cyclic(rs)
rs = add_lags(rs)
rs = add_seasonal_lags(rs)
rs = add_moving_averages(rs)
rs = add_weather_data(rs,wd)
rs = add_more_weather_feature(rs)
rs = add_federal_holidays(rs, custom_holidays = ['12-31'])
rs = add_law_flag(rs, law_name='Trash_Law', start_date = '2024-03-01')
rs = add_law_flag(rs, law_name = 'New_Trash_Law', start_date = '2024-11-01')
rs = add_law_flag(rs, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
rs = add_law_flag(rs, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

In [36]:
rs

,y,dayofweek,quarter,month,year,dayofyear,dayofmonth,weekofyear,dayofweek_sin,dayofweek_cos,...,apparent_temperature_min_lag365,apparent_temperature_min_lag730,temperature_2m_max_lag14,temperature_2m_max_lag30,temperature_2m_max_lag60,is_federal_holiday,Trash_Law,New_Trash_Law,Rat_Mitigation_Zone,Rat_Czar_Appointed
ds,,,,,,,,,,,,,,,,,,,,,
2020-01-01,17,2,1,1,2020,1,1,1,0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,1,0,0,0,0
2020-01-02,40,3,1,1,2020,2,2,1,0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-03,41,4,1,1,2020,3,3,1,-0.433884,-0.900969,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-04,25,5,1,1,2020,4,4,1,-0.974928,-0.222521,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
2020-01-05,17,6,1,1,2020,5,5,1,-0.781831,0.623490,...,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-02-24,26,1,1,2,2026,55,24,9,0.781831,0.623490,...,-4.6,-11.1,-0.3,-5.2,-2.8,0,1,1,1,1
2026-02-25,30,2,1,2,2026,56,25,9,0.974928,-0.222521,...,0.1,-6.9,3.2,-3.1,-0.8,0,1,1,1,1
2026-02-26,40,3,1,2,2026,57,26,9,0.433884,-0.900969,...,-1.4,-4.1,0.8,-7.3,1.3,0,1,1,1,1


### Features for XGBoost

In [37]:
FEATURES = ['apparent_temperature_min_lag30',
            'apparent_temperature_min_lag60',
            'apparent_temperature_min_lag120',
            'apparent_temperature_min_lag365',
            'apparent_temperature_min_lag730',
            'dayofyear', 
            'temperature_2m_max_lag30',
            'temperature_2m_max_lag60'
            ]

### Add default parameters for XGBoost

In [38]:
params = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
         'base_score': 0.5, 
         'n_estimators': 1000, 
        #  'min_child_weight': 6, 
        # 'learning_rate': 0.01,
        # 'max_depth': 8, 
        #  'subsample': 1,
        #  'colsample_bytree': 0.96,
        #  'colsample_bylevel': 0.6, 
        #  'colsample_bynode': 0.9, 
        #  'reg_alpha': 2.2, 
        #  'gamma': 100, 
        #  'reg_lambda': 0.18,
        #  'early_stopping_rounds': 100, 
        }


### Results for XGBoost Model

In [39]:
print(FEATURES)
print(params)
TARGET = 'y'

# Gotta make sure the features and parameters exist.

reg = xgb.XGBRegressor(**params)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    reg.fit(train[FEATURES], train[TARGET])
    y_pred = reg.predict(test[FEATURES])
    y_true = test[TARGET].values
    
    # Our metrics
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

xgb_results_df = pd.DataFrame(results)
mean_rmse = xgb_results_df['rmse'].mean()
mean_mape = xgb_results_df['mape'].mean()
xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

['apparent_temperature_min_lag30', 'apparent_temperature_min_lag60', 'apparent_temperature_min_lag120', 'apparent_temperature_min_lag365', 'apparent_temperature_min_lag730', 'dayofyear', 'temperature_2m_max_lag30', 'temperature_2m_max_lag60']
{'objective': 'reg:squarederror', 'eval_metric': 'rmse', 'booster': 'gbtree', 'base_score': 0.5, 'n_estimators': 1000}


In [40]:
xgb_results_df

,fold,rmse,mape
0,0,19.929296,0.378072
1,1,17.512622,0.265612
2,2,20.054394,0.301435
3,3,27.480547,0.434034
4,4,25.262211,0.360204
5,5,23.180691,0.325691
6,6,27.919958,0.351842
7,7,28.444858,0.385875
8,8,22.450725,0.320594
9,9,28.364353,0.361542


## XGBoosted Prophet Model

In [41]:
## Recall the copy that was saved. 
## This was because Prophet demanded a particular format while SARIMA and Holt-Winters didn't.
rs_saved
rs = rs_saved

In [42]:
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    # Split the dataset into training and testing sets
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # Fit Prophet on the training data
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)
    
    # Make predictions on the training set to calculate residuals
    train_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
    train_forecast = model.predict(train_future)
    
    # Calculate residuals (actual - predicted) on the training data
    train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    
    # Build a new DataFrame of residuals
    residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals })
    # Add the features I defined above
    train = train.set_index('ds')
    train.index = pd.to_datetime(train.index)
    train = create_features(train)
    train = add_cyclic(train)
    train = add_lags(train)
    train = add_seasonal_lags(train)
    train = add_moving_averages(train)
    train = add_weather_data(train,wd)
    train = add_more_weather_feature(train)
    train = add_federal_holidays(train, custom_holidays = ['12-31'])
    train = add_law_flag(train, law_name='Trash_Law', start_date = '2024-03-01')
    train = add_law_flag(train, law_name = 'New_Trash_Law', start_date = '2024-11-01')
    train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
    train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

    X_train_residuals = train[FEATURES]
    y_train_residuals = residuals_df['y']
    
    xgb_model = xgb.XGBRegressor(**params)
    xgb_model.fit(X_train_residuals, y_train_residuals)
    
    # Add the features I defined above (I should probably define a new function to do this in one go...)
    test = test.set_index('ds')
    test.index = pd.to_datetime(test.index)
    test = create_features(test)
    test = add_cyclic(test)
    test = add_lags(test)
    test = add_seasonal_lags(test)
    test = add_moving_averages(test)
    test = add_weather_data(test,wd)
    test = add_more_weather_feature(test)
    test = add_federal_holidays(test, custom_holidays = ['12-31'])
    test = add_law_flag(test, law_name='Trash_Law', start_date = '2024-03-01')
    test = add_law_flag(test, law_name = 'New_Trash_Law', start_date = '2024-11-01')
    test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
    test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

    # Predict residuals using XGBoost for the test set
    X_test = test[FEATURES]  # Features for the test set
    xgb_residual_preds = xgb_model.predict(X_test)
    
    # Forecast using Prophet on the test set
    future = model.make_future_dataframe(periods=len(test), freq='D')
    prophet_forecast = model.predict(future)
    
    # Combine Prophet's forecast and XGBoost's residual prediction
    y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    y_true = test['y'].values
    
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Store the results for this fold
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})
    
    
    ## Uncomment code below if you want to have plots on feature importance. I'll leave it commented out for obvious reasons.
    # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
    # plot_importance(xgb_model, ax=ax1, importance_type='gain')
    # ax1.set_title('Gain-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax2, importance_type='weight')
    # ax2.set_title('Split-based Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax3, importance_type='cover')
    # ax3.set_title('Cover Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
    # ax4.set_title('Total Gain Importance', fontsize=12)

    # plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
    # ax5.set_title('Total Cover Importance', fontsize=12)

    plt.show()

# Convert the results into a DataFrame
prophet_xgb_results_df = pd.DataFrame(results)
mean_rmse = prophet_xgb_results_df['rmse'].mean()
mean_mape = prophet_xgb_results_df['mape'].mean()
prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

01:35:28 - cmdstanpy - INFO - Chain [1] start processing
01:35:28 - cmdstanpy - INFO - Chain [1] done processing
01:35:30 - cmdstanpy - INFO - Chain [1] start processing
01:35:30 - cmdstanpy - INFO - Chain [1] done processing
01:35:33 - cmdstanpy - INFO - Chain [1] start processing
01:35:33 - cmdstanpy - INFO - Chain [1] done processing
01:35:35 - cmdstanpy - INFO - Chain [1] start processing
01:35:35 - cmdstanpy - INFO - Chain [1] done processing
01:35:37 - cmdstanpy - INFO - Chain [1] start processing
01:35:38 - cmdstanpy - INFO - Chain [1] done processing
01:35:40 - cmdstanpy - INFO - Chain [1] start processing
01:35:40 - cmdstanpy - INFO - Chain [1] done processing
01:35:42 - cmdstanpy - INFO - Chain [1] start processing
01:35:42 - cmdstanpy - INFO - Chain [1] done processing
01:35:44 - cmdstanpy - INFO - Chain [1] start processing
01:35:44 - cmdstanpy - INFO - Chain [1] done processing
01:35:47 - cmdstanpy - INFO - Chain [1] start processing
01:35:47 - cmdstanpy - INFO - Chain [1]

In [43]:
prophet_xgb_results_df

,fold,rmse,mape
0,0,13.215907,0.231255
1,1,10.566206,0.165001
2,2,7.966207,0.126356
3,3,9.757844,0.141595
4,4,10.479492,0.128999
5,5,19.946150,0.297640
6,6,12.307890,0.138567
7,7,12.295732,0.137834
8,8,11.804650,0.130837
9,9,9.067968,0.114356


# Conclusions on Model Comparisons

## Results Table

In [44]:
# We make a dictionary of models and their results to make it easier to iterate over. 
# Make sure to add to this if writing new models in.
models = {
    'baseline': baseline_results_df,
    'rolling4w': rolling4w_results_df,
    'prophet': prophet_results_df,
    'sarima': sarima_results_df,
    'hw': hw_results_df,
    'xgb': xgb_results_df,
    'prophet+xgb': prophet_xgb_results_df
}

all_results = []
for model_name, df in models.items():
    df['model'] = model_name
    all_results.append(df)

# Put all of the dataframes together into one dataframe for display
final_results_df = pd.concat(all_results, ignore_index=True)
# Make a pivot table so that we display rmse, mape and then each of the models and their results.
final_table = final_results_df.pivot(index='fold', columns='model', values=['rmse', 'mape'])
final_table.index = final_table.index.where(final_table.index != '-', 'mean')

final_table

rmse                                                          \
model   baseline         hw    prophet prophet+xgb  rolling4w     sarima   
fold                                                                       
0      15.307003  17.029947  12.931175   13.215907  14.323744  15.484218   
1      11.914386  17.558940   9.286185   10.566206  12.719150  10.325060   
2      14.988228  17.312295   7.983666    7.966207  12.398157  10.673432   
3      20.818662  22.556060   9.817654    9.757844  19.240721  13.541181   
4      17.563835  19.966872  10.903003   10.479492  15.923646  13.465575   
5      23.244116  20.725658  15.534582   19.946150  17.378918  15.979406   
6      19.488233  22.555124  11.539387   12.307890  24.564601  14.095096   
7      21.222927  25.834040  11.299325   12.295732  20.989793  17.213794   
8      20.494910  23.343693  10.559222   11.804650  19.858428  16.312946   
9      21.193296  21.316698   9.146496    9.067968  18.798509  10.247811   
10     31.447761  23.475082  15.403790   12.017919  15.939618  14.885509   
11     22.102668  20.733607   9.164685   12.556884  13.952726  10.975919   
12     23.922867  18.607673  11.141143   12.558643  17.455044  15.912922   
13     19.713532  26.490924  12.065524   13.895760  19.822201  16.129220   
14     34.114645  26.457852  18.946567   14.295062  21.751026  22.762945   
15     26.451596  17.653667   8.367090    6.394346  13.761683  10.133942   
16     28.910678  15.214237  11.798670   15.303415  12.954591   7.546025   
17     35.881458  16.538625  15.583406   31.750496  19.309278  13.890276   
18     24.802787  13.508845  11.766219   14.646116   8.641676  14.015057   
19     18.947508  17.519674  11.907598   16.534576  15.194865  13.129162   
20     24.849791  11.076713  10.669815   11.465491  14.811916   7.397422   
21     22.731809  12.828411   9.164783   15.065641   8.280248   9.809125   
22     20.216302  13.621282   9.808565   15.832526  15.873383  10.320049   
23     30.933127  15.432451  11.468369   20.537240  11.649647  11.549283   
24     30.349771   6.899231   9.018752    9.224914   8.625129   4.775254   
25     26.045942  14.060919  14.222347   14.096187  13.177023  10.821048   
mean   23.371455  18.396866  11.519154   13.599356  15.669066  12.745834   

                      mape                                            \
model        xgb  baseline        hw   prophet prophet+xgb rolling4w   
fold                                                                   
0      19.929296  0.285977  0.279665  0.246172    0.231255  0.239272   
1      17.512622  0.186347  0.255894  0.142636    0.165001  0.196755   
2      20.054394  0.250230  0.297496  0.130110    0.126356  0.196394   
3      27.480547  0.348457  0.358384  0.143238    0.141595  0.260374   
4      25.262211  0.256820  0.268691  0.138791    0.128999  0.220183   
5      23.180691  0.361006  0.310186  0.217436    0.297640  0.256428   
6      27.919958  0.266129  0.270105  0.145670    0.138567  0.247419   
7      28.444858  0.284166  0.311909  0.135217    0.137834  0.255484   
8      22.450725  0.301036  0.306820  0.118595    0.130837  0.253080   
9      28.364353  0.277230  0.274190  0.113947    0.114356  0.187194   
10     35.889021  0.498262  0.358957  0.226627    0.173974  0.229220   
11     25.697181  0.317831  0.281012  0.116819    0.172471  0.194994   
12     24.447045  0.359283  0.213410  0.119338    0.167101  0.200704   
13     18.542184  0.275826  0.314054  0.120530    0.137875  0.223940   
14     33.376602  0.766270  0.575931  0.387667    0.285858  0.478330   
15     25.173319  0.582483  0.338866  0.157208    0.119550  0.290997   
16     28.274260  0.723483  0.350417  0.256426    0.328142  0.308417   
17     28.957837  1.485229  0.650859  0.553277    1.196906  0.808842   
18     19.730613  0.767713  0.299738  0.308413    0.373153  0.170184   
19     19.677343  0.787489  0.500986  0.365745    0.564656  0.628196   
20     18.549521  1.259635  0.512212  0.409370    0.475227  0.750920   
21 

## Summary

In the above table, we see that the Prophet model had the best average RMSE. We also see that the next best were the SARIMA model and the Prophet + XGBoost models. Therefore, we will select the Prophet model for our modeling purposes and only maybe consider training a Prophet + XGBoost model to see if there is any improvement.

We now know that the Prophet model performs the best, but we might ask if using an XGBoost model to predict residuals can lead to better model performance. In the above, we were unable to do that, but we were also using suboptimal parameters. 

# Comparing Prophet Model versus an XGBoosted Prophet Model

This section of code can be long so we summarize. First, we saw from before that Prophet performed very well. We also saw that Prophet + XGBoost on residuals performed slightly worse. This could be due to not tuning the parameters very well. So, we might instead try to see if we can get better predictions by actually doing some feature engineering and hyperparameter tuning of XGBoost.

There are a few things we must be careful about.

1. Remember that we are evaluating based on walk-forward validation. That means we fix a training set, and then evaluate 14 days out. Then for the next fold, we add those 14 days to our training set and evaluate 14 days out again. Then add those 14 days to the training set and evaluate 14 days out again. Then we repeat until we have done 26 folds.

2. Suppose we fix a fold so we have a training set and a test set decided. We first fit Prophet to the training set. Then determine the residuals there. Using the residuals, we then train an XGBoost model fitted to the residuals. Then, we use both the Prophet model and the XGBoost model to forecast 14 days out and we evaluate against the test set.

3. We want **clear** evidence of improvement. So, we had proceeded as before and used a TimeSeriesSplit. In training the XGBoost model, we need to be careful of overfitting. We use Optuna to tune the hyperparameters and manually select features until we see improvement. The benefit of using Optuna is that we have written our objective function to avoid overfitting to the residuals' training set. 

4. Even if we have *clear* evidence of improvement, we also must ensure that the added **complexity** is worth it. Training an XGBoost model is quite time intensive and having to tune hyperparameters for each fold is already a lot of work. The value of Optuna here is that we can limit the number of trials.

## Reimport the Data

We reimport the data so that we can focus only on running this section's code as opposed to having to rerun from the start.

In [45]:
tscv = TimeSeriesSplit(gap=0, max_train_size=None, n_splits=26, test_size=14)

In [46]:
rs = pd.read_csv('../scr/data/cleaned_rat_sightings_data/all_cleaned_rat_sightings.csv')
rs['created_date'] = pd.to_datetime(rs['created_date']) 
rs['closed_date'] = pd.to_datetime(rs['closed_date'])
rs['resolution_action_updated_date'] = pd.to_datetime(rs['resolution_action_updated_date'])

In [47]:
# Start by cutting off data before 2020-01-01 and after 2026-02-28.
rs = rs[rs['created_date']<'2026-03-01']
rs = rs[rs['created_date']>='2020-01-01']

In [48]:
rs = rs.groupby([rs['created_date'].dt.date]).size().reset_index(name='count')

In [49]:
## This is 2251 which equals the number of days from 2020-01-01 to 2026-02-28
len(rs)

2251

In [50]:
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)

In [51]:
rs

,ds,y
0,2020-01-01,17
1,2020-01-02,40
2,2020-01-03,41
3,2020-01-04,25
4,2020-01-05,17
...,...,...
2246,2026-02-24,26
2247,2026-02-25,30
2248,2026-02-26,40
2249,2026-02-27,38


## Repeat the Computations for Prophet

In [52]:
# Create a date range covering 2020 through end of 2025
date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# Build the DataFrame in the same structure as your original
federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1,
})

holidays = federal_holidays

# Rename columns for Prophet model
rs.rename(columns={'created_date': 'ds', 'count': 'y'}, inplace=True)
results = []

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')

    model.fit(train)
    
    future = model.make_future_dataframe(periods=len(test), freq='D')
    forecast = model.predict(future)
    
    # Obtain predicted values and compare against the actuals.
    y_pred = forecast['yhat'][-len(test):].values
    y_true = test['y'].values
    
    # Calculate RMSE
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
    # Calculate MAPE
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    # Append results
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Convert results to a datafrane
prophet_results_df = pd.DataFrame(results)

prophet_results_df.loc['mean'] = ['mean',  prophet_results_df['rmse'].mean(), prophet_results_df['mape'].mean()]

01:36:32 - cmdstanpy - INFO - Chain [1] start processing
01:36:33 - cmdstanpy - INFO - Chain [1] done processing
01:36:33 - cmdstanpy - INFO - Chain [1] start processing
01:36:33 - cmdstanpy - INFO - Chain [1] done processing
01:36:34 - cmdstanpy - INFO - Chain [1] start processing
01:36:34 - cmdstanpy - INFO - Chain [1] done processing
01:36:34 - cmdstanpy - INFO - Chain [1] start processing
01:36:35 - cmdstanpy - INFO - Chain [1] done processing
01:36:35 - cmdstanpy - INFO - Chain [1] start processing
01:36:35 - cmdstanpy - INFO - Chain [1] done processing
01:36:36 - cmdstanpy - INFO - Chain [1] start processing
01:36:36 - cmdstanpy - INFO - Chain [1] done processing
01:36:36 - cmdstanpy - INFO - Chain [1] start processing
01:36:36 - cmdstanpy - INFO - Chain [1] done processing
01:36:37 - cmdstanpy - INFO - Chain [1] start processing
01:36:37 - cmdstanpy - INFO - Chain [1] done processing
01:36:37 - cmdstanpy - INFO - Chain [1] start processing
01:36:37 - cmdstanpy - INFO - Chain [1]

## Adding Features to XGBoost

In [53]:
def create_features(df):
    """
    Create time series features based on time series index.
    """
    df = df.copy()
    df['dayofweek'] = df.index.dayofweek
    df['quarter'] = df.index.quarter
    df['month'] = df.index.month
    df['year'] = df.index.year
    df['dayofyear'] = df.index.dayofyear
    df['dayofmonth'] = df.index.day
    df['weekofyear'] = df.index.isocalendar().week
    return df

def add_cyclic(df):
    target_map = df['y'].to_dict()
    df['dayofweek_sin'] = np.sin(2 * np.pi * df['dayofweek']/7)
    df['dayofweek_cos'] = np.cos(2 * np.pi * df['dayofweek']/7)
    df['month_sin'] = np.sin(2 * np.pi * df['month']/12)
    df['month_cos'] = np.cos(2 * np.pi * df['month']/12)
    return df

def add_lags(df):
    target_map = df['y'].to_dict()
    df['lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['lag2'] = (df.index - pd.Timedelta('2 days')).map(target_map)
    df['lag3'] = (df.index - pd.Timedelta('3 days')).map(target_map)
    df['lag4'] = (df.index - pd.Timedelta('4 days')).map(target_map)
    df['lag5'] = (df.index - pd.Timedelta('5 days')).map(target_map)
    df['lag6'] = (df.index - pd.Timedelta('6 days')).map(target_map)
    df['lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['lag8'] = (df.index - pd.Timedelta('8 days')).map(target_map)
    df['lag9'] = (df.index - pd.Timedelta('9 days')).map(target_map)
    df['lag10'] = (df.index - pd.Timedelta('10 days')).map(target_map)
    df['lag11'] = (df.index - pd.Timedelta('11 days')).map(target_map)
    df['lag12'] = (df.index - pd.Timedelta('12 days')).map(target_map)
    df['lag13'] = (df.index - pd.Timedelta('13 days')).map(target_map)
    df['lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    return df

def add_seasonal_lags(df):
    target_map = df['y'].to_dict()
    df['lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)

    df['lag362'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag363'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag364'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag366'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['lag367'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    
    df['lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)
    df['lag1095'] = (df.index - pd.Timedelta('1095 days')).map(target_map)
    df['lag1460'] = (df.index - pd.Timedelta('1460 days')).map(target_map)
    df['lag1825'] = (df.index - pd.Timedelta('1825 days')).map(target_map)
    return df

def add_moving_averages(df):
    df = df.copy()
    
    # Ensure sorted by date
    df = df.sort_index()
    
    # Moving averages (using previous values only)
    df['ma7'] = df['y'].shift(1).rolling(window=7).mean()
    df['ma30'] = df['y'].shift(1).rolling(window=30).mean()
    df['ma60'] = df['y'].shift(1).rolling(window=60).mean()
    df['ma90'] = df['y'].shift(1).rolling(window=90).mean()
    df['ma120'] = df['y'].shift(1).rolling(window=120).mean()
    df['ma150'] = df['y'].shift(1).rolling(window=150).mean()
    df['ma180'] = df['y'].shift(1).rolling(window=180).mean()
    df['ma365'] = df['y'].shift(1).rolling(window=365).mean()
    
    return df


In the next two code block, we add weather data to the data set. This is not optimized i.e. we just obtain the weather data in Manhattan and hope that it is representative of the average weather over the whole city.

In [54]:
## Add weather data.

import requests
import pandas as pd

lat, lon = 40.7831, -73.9712
start = "2020-01-01"
end   = "2026-02-28"

url = (
    "https://archive-api.open-meteo.com/v1/archive"
    f"?latitude={lat}&longitude={lon}"
    f"&start_date={start}&end_date={end}"
    "&daily=temperature_2m_max,temperature_2m_min,temperature_2m_mean,"
    "apparent_temperature_max,apparent_temperature_min,apparent_temperature_mean,"
    "precipitation_sum,snowfall_sum"
    "&timezone=America/New_York"
)

response = requests.get(url)
data = response.json()

if 'error' in data:
    nd = pd.read_csv("weatherdata.csv")
    nd = nd.set_index('date')
    wd = nd

else:
    wd = pd.DataFrame(data["daily"])
    wd["date"] = pd.to_datetime(wd["time"])
    wd = wd.set_index("date")

In [55]:
def add_weather_data(df, wd):
    df = df.copy()
    wd = wd.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    wd.index = pd.to_datetime(wd.index)
    
    # Drop unnecessary column
    if "time" in wd.columns:
        wd = wd.drop(columns=["time"])
    
    # Join on date index
    df = df.join(wd, how="left")
    
    return df

In [56]:
from pandas.tseries.holiday import USFederalHolidayCalendar

def add_federal_holidays(df, custom_holidays=None):
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    cal = USFederalHolidayCalendar()
    holidays = cal.holidays(start=df.index.min(), end=df.index.max())
    
    if custom_holidays:
        for d in custom_holidays:
            if len(d) == 5:  # MM-DD format → recurring annually
                years = df.index.year.unique()
                for y in years:
                    holidays = holidays.append(pd.to_datetime([f"{y}-{d}"]))
            else:  # YYYY-MM-DD → one specific date
                holidays = holidays.append(pd.to_datetime([d]))
    
    holidays = holidays.drop_duplicates().sort_values()
    
    df["is_federal_holiday"] = df.index.isin(holidays).astype(int)
    
    return df

In [57]:
def add_law_flag(df, law_name: str, start_date: str):
    # Adds a binary column to indicate when a new law is active.
    df = df.copy()
    
    # Ensure datetime index
    df.index = pd.to_datetime(df.index)
    
    # Convert start_date to datetime
    start_dt = pd.to_datetime(start_date)
    
    # Create binary column: 1 if date >= start_date, else 0
    df[law_name] = (df.index >= start_dt).astype(int)
    
    return df

In [58]:
# This must be run importing weather data

def add_more_weather_feature(df):
    target_map = df['apparent_temperature_min'].to_dict()
    df['apparent_temperature_min_lag1'] = (df.index - pd.Timedelta('1 days')).map(target_map)
    df['apparent_temperature_min_lag7'] = (df.index - pd.Timedelta('7 days')).map(target_map)
    df['apparent_temperature_min_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['apparent_temperature_min_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['apparent_temperature_min_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)
    df['apparent_temperature_min_lag90'] = (df.index - pd.Timedelta('90 days')).map(target_map)
    df['apparent_temperature_min_lag120'] = (df.index - pd.Timedelta('120 days')).map(target_map)
    df['apparent_temperature_min_lag150'] = (df.index - pd.Timedelta('150 days')).map(target_map)
    df['apparent_temperature_min_lag180'] = (df.index - pd.Timedelta('180 days')).map(target_map)
    df['apparent_temperature_min_lag210'] = (df.index - pd.Timedelta('210 days')).map(target_map)
    df['apparent_temperature_min_lag240'] = (df.index - pd.Timedelta('240 days')).map(target_map)
    df['apparent_temperature_min_lag270'] = (df.index - pd.Timedelta('270 days')).map(target_map)
    df['apparent_temperature_min_lag300'] = (df.index - pd.Timedelta('300 days')).map(target_map)
    df['apparent_temperature_min_lag330'] = (df.index - pd.Timedelta('330 days')).map(target_map)
    df['apparent_temperature_min_lag360'] = (df.index - pd.Timedelta('360 days')).map(target_map)
    df['apparent_temperature_min_lag365'] = (df.index - pd.Timedelta('365 days')).map(target_map)
    df['apparent_temperature_min_lag730'] = (df.index - pd.Timedelta('730 days')).map(target_map)

    target_map = df['temperature_2m_max'].to_dict()
    df['temperature_2m_max_lag14'] = (df.index - pd.Timedelta('14 days')).map(target_map)
    df['temperature_2m_max_lag30'] = (df.index - pd.Timedelta('30 days')).map(target_map)
    df['temperature_2m_max_lag60'] = (df.index - pd.Timedelta('60 days')).map(target_map)

    return df

## Features for XGBoost

In [59]:
FEATURES = ['apparent_temperature_min_lag30',
            'apparent_temperature_min_lag60',
            'apparent_temperature_min_lag120',
            'apparent_temperature_min_lag365',
            'apparent_temperature_min_lag730',
            'dayofyear', 
            'temperature_2m_max_lag30',
            'temperature_2m_max_lag60'
            ]

## Add default parameters for XGBoost

In [60]:
params = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
         'base_score': 0.5, 
         'n_estimators': 1000, 
        #  'min_child_weight': 6, 
        # 'learning_rate': 0.01,
        # 'max_depth': 8, 
        #  'subsample': 1,
        #  'colsample_bytree': 0.96,
        #  'colsample_bylevel': 0.6, 
        #  'colsample_bynode': 0.9, 
        #  'reg_alpha': 2.2, 
        #  'gamma': 100, 
        #  'reg_lambda': 0.18,
        #  'early_stopping_rounds': 100, 
        }


In [61]:
## Old code to evaluate XGBoost + Prophet model.

# results = []

# for i, (train_index, test_index) in enumerate(tscv.split(rs)):
#     # Split the dataset into training and testing sets
#     train = rs.iloc[train_index]
#     test = rs.iloc[test_index]
    
#     # Fit Prophet on the training data
#     model = Prophet(holidays=holidays)
#     model.add_country_holidays(country_name='US')
#     model.fit(train)
    
#     # Make predictions on the training set to calculate residuals
#     train_future = model.make_future_dataframe(periods=0, freq='D')  # Use periods=0 to only use the training data
#     train_forecast = model.predict(train_future)
    
#     # Calculate residuals (actual - predicted) on the training data
#     train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    
#     # Build a new DataFrame of residuals
#     residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals })

#     train = train.set_index('ds')
#     train.index = pd.to_datetime(train.index)
#     train = create_features(train)
#     train = add_cyclic(train)
#     train = add_lags(train)
#     train = add_seasonal_lags(train)
#     train = add_moving_averages(train)
#     train = add_weather_data(train,wd)
#     train = add_more_weather_feature(train)
#     train = add_federal_holidays(train, custom_holidays = ['12-31'])
#     train = add_law_flag(train, law_name='Trash_Law', start_date = '2024-03-01')
#     train = add_law_flag(train, law_name = 'New_Trash_Law', start_date = '2024-11-01')
#     train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
#     train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

#     X_train_residuals = train[FEATURES]
#     y_train_residuals = residuals_df['y']

    
    
#     xgb_model = xgb.XGBRegressor(**params)
#     xgb_model.fit(X_train_residuals, y_train_residuals)
    

#     test = test.set_index('ds')
#     test.index = pd.to_datetime(test.index)
#     test = create_features(test)
#     test = add_cyclic(test)
#     test = add_lags(test)
#     test = add_seasonal_lags(test)
#     test = add_moving_averages(test)
#     test = add_weather_data(test,wd)
#     test = add_more_weather_feature(test)
#     test = add_federal_holidays(test, custom_holidays = ['12-31'])
#     test = add_law_flag(test, law_name='Trash_Law', start_date = '2024-03-01')
#     test = add_law_flag(test, law_name = 'New_Trash_Law', start_date = '2024-11-01')
#     test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date = '2023-07-07')
#     test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date = '2023-04-12')

#     # Predict residuals using XGBoost for the test set
#     X_test = test[FEATURES]  # Features for the test set
#     xgb_residual_preds = xgb_model.predict(X_test)
    
#     # Forecast using Prophet on the test set
#     future = model.make_future_dataframe(periods=len(test), freq='D')
#     prophet_forecast = model.predict(future)
    
#     # Combine Prophet's forecast and XGBoost's residual prediction
#     y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    
#     # Actual values for the test set
#     y_true = test['y'].values
    
#     # Calculate RMSE
#     rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    
#     # Calculate MAPE
#     mape = mean_absolute_percentage_error(y_true, y_pred)
    
#     # Store the results for this fold
#     results.append({'fold': i, 'rmse': rmse, 'mape': mape})
    
    
#     ## Uncomment code below if you want to have plots on feature importance.
#     # fig, (ax1, ax2, ax3, ax4, ax5) = plt.subplots(5, 1, figsize=(10, 30))
#     # plot_importance(xgb_model, ax=ax1, importance_type='gain')
#     # ax1.set_title('Gain-based Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax2, importance_type='weight')
#     # ax2.set_title('Split-based Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax3, importance_type='cover')
#     # ax3.set_title('Cover Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax4, importance_type='total_gain')
#     # ax4.set_title('Total Gain Importance', fontsize=12)

#     # plot_importance(xgb_model, ax=ax5, importance_type='total_cover')
#     # ax5.set_title('Total Cover Importance', fontsize=12)

    
#     plt.show()

# # Convert the results into a DataFrame
# prophet_xgb_results_df = pd.DataFrame(results)

# prophet_xgb_results_df = pd.DataFrame(results)
# mean_rmse = prophet_xgb_results_df['rmse'].mean()
# mean_mape = prophet_xgb_results_df['mape'].mean()
# prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

## Optuna tuning before each fold for best hyperparameters

The following is a very long code block and probably takes quite a long time to run so I've tried my best to break down the workings of the code with tons of comments.

In [62]:
import optuna
from sklearn.model_selection import train_test_split

results = []

def objective(trial, X_train_residuals, y_train_residuals):
    param = {'objective': 'reg:squarederror',
         'eval_metric': 'rmse',
         'booster': 'gbtree',
        'n_estimators': trial.suggest_int('n_estimators', 700, 1000),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'min_child_weight': trial.suggest_int('min_child_weight', 3, 10),
        #'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        #'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        #'gamma': trial.suggest_float('gamma', 0, 5),
        #'reg_alpha': trial.suggest_float('reg_alpha', 0, 5),
        #'reg_lambda': trial.suggest_float('reg_lambda', 0, 5),
        'random_state': 42
    }

    tscv_inner = TimeSeriesSplit(n_splits=5)  # small CV on fold

    rmses = []
    for tr_idx, val_idx in tscv_inner.split(X_train_residuals):
        X_tr, X_val = X_train_residuals.iloc[tr_idx], X_train_residuals.iloc[val_idx]
        y_tr, y_val = y_train_residuals.iloc[tr_idx], y_train_residuals.iloc[val_idx]

        model = xgb.XGBRegressor(**param)
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))

    return np.mean(rmses)

for i, (train_index, test_index) in enumerate(tscv.split(rs)):
    # Split the dataset
    train = rs.iloc[train_index]
    test = rs.iloc[test_index]
    
    # Fit Prophet
    model = Prophet(holidays=holidays)
    model.add_country_holidays(country_name='US')
    model.fit(train)
    
    # Calculate residuals
    train_future = model.make_future_dataframe(periods=0, freq='D')
    train_forecast = model.predict(train_future)
    train_residuals = train['y'].values - train_forecast['yhat'][:len(train)].values
    residuals_df = pd.DataFrame({'ds': train['ds'], 'y': train_residuals})
    
    # Feature engineering
    train = train.set_index('ds')
    train.index = pd.to_datetime(train.index)
    train = create_features(train)
    train = add_cyclic(train)
    train = add_lags(train)
    train = add_seasonal_lags(train)
    train = add_moving_averages(train)
    train = add_weather_data(train, wd)
    train = add_more_weather_feature(train)
    train = add_federal_holidays(train, custom_holidays=['12-31'])
    train = add_law_flag(train, law_name='Trash_Law', start_date='2024-03-01')
    train = add_law_flag(train, law_name='New_Trash_Law', start_date='2024-11-01')
    train = add_law_flag(train, law_name='Rat_Mitigation_Zone', start_date='2023-07-07')
    train = add_law_flag(train, law_name='Rat_Czar_Appointed', start_date='2023-04-12')

    X_train_residuals = train[FEATURES]
    y_train_residuals = residuals_df['y']

    best_params = params
    # Uncomment the next 4 lines to include Optuna hyperparameter tuning. We used Optuna because our parameter space is quite large.
    # study = optuna.create_study(direction='minimize')
    # study.optimize(lambda trial: objective(trial, X_train_residuals, y_train_residuals), n_trials=5)
    # best_params = study.best_params 
    # best_params['random_state'] = 42

    # Train XGBoost with best parameters from Optuna.
    xgb_model = xgb.XGBRegressor(**best_params)
    xgb_model.fit(X_train_residuals, y_train_residuals)
    
    # Add the features we will use for XGBoost. Again, we are using XGBoost to try and predict the residuals of the Prophet model.
    test = test.set_index('ds')
    test.index = pd.to_datetime(test.index)
    test = create_features(test)
    test = add_cyclic(test)
    test = add_lags(test)
    test = add_seasonal_lags(test)
    test = add_moving_averages(test)
    test = add_weather_data(test, wd)
    test = add_more_weather_feature(test)
    test = add_federal_holidays(test, custom_holidays=['12-31'])
    test = add_law_flag(test, law_name='Trash_Law', start_date='2024-03-01')
    test = add_law_flag(test, law_name='New_Trash_Law', start_date='2024-11-01')
    test = add_law_flag(test, law_name='Rat_Mitigation_Zone', start_date='2023-07-07')
    test = add_law_flag(test, law_name='Rat_Czar_Appointed', start_date='2023-04-12')

    # Predict the residuals of Prophet for test set
    X_test = test[FEATURES]
    xgb_residual_preds = xgb_model.predict(X_test)
    
    # Have prophet make a forecast on the test set which is 14 days.
    future = model.make_future_dataframe(periods=len(test), freq='D')
    prophet_forecast = model.predict(future)
    
    # Add the forecast of the residuals by XGBoost for 14 days.
    y_pred = prophet_forecast['yhat'][-len(test):].values + xgb_residual_preds
    y_true = test['y'].values

    # Obtain metrics for our forecast against the actuals.
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mape = mean_absolute_percentage_error(y_true, y_pred)
    
    results.append({'fold': i, 'rmse': rmse, 'mape': mape})

# Results DataFrame
prophet_xgb_results_df = pd.DataFrame(results)
mean_rmse = prophet_xgb_results_df['rmse'].mean()
mean_mape = prophet_xgb_results_df['mape'].mean()
prophet_xgb_results_df.loc['mean'] = ['mean', mean_rmse, mean_mape]

01:36:49 - cmdstanpy - INFO - Chain [1] start processing
01:36:50 - cmdstanpy - INFO - Chain [1] done processing
01:36:52 - cmdstanpy - INFO - Chain [1] start processing
01:36:52 - cmdstanpy - INFO - Chain [1] done processing
01:36:54 - cmdstanpy - INFO - Chain [1] start processing
01:36:54 - cmdstanpy - INFO - Chain [1] done processing
01:36:56 - cmdstanpy - INFO - Chain [1] start processing
01:36:56 - cmdstanpy - INFO - Chain [1] done processing
01:36:58 - cmdstanpy - INFO - Chain [1] start processing
01:36:58 - cmdstanpy - INFO - Chain [1] done processing
01:37:00 - cmdstanpy - INFO - Chain [1] start processing
01:37:01 - cmdstanpy - INFO - Chain [1] done processing
01:37:03 - cmdstanpy - INFO - Chain [1] start processing
01:37:03 - cmdstanpy - INFO - Chain [1] done processing
01:37:05 - cmdstanpy - INFO - Chain [1] start processing
01:37:05 - cmdstanpy - INFO - Chain [1] done processing
01:37:08 - cmdstanpy - INFO - Chain [1] start processing
01:37:08 - cmdstanpy - INFO - Chain [1]

## One Last Comparison

In [63]:
prophet_xgb_results_df

,fold,rmse,mape
0,0,13.215907,0.231255
1,1,10.566206,0.165001
2,2,7.966207,0.126356
3,3,9.757844,0.141595
4,4,10.479492,0.128999
5,5,19.946150,0.297640
6,6,12.307890,0.138567
7,7,12.295732,0.137834
8,8,11.804650,0.130837
9,9,9.067968,0.114356


In [64]:
prophet_results_df

,fold,rmse,mape
0,0,12.931175,0.246172
1,1,9.286185,0.142636
2,2,7.983666,0.130110
3,3,9.817654,0.143238
4,4,10.903003,0.138791
5,5,15.534582,0.217436
6,6,11.539387,0.145670
7,7,11.299325,0.135217
8,8,10.559222,0.118595
9,9,9.146496,0.113947


# Prophet Model for Daily Rat Sightings in New York City

For ease, we reimport the data. This way, we can run only the code in this section as opposed to having to run the code blocks above.

In [65]:
from prophet.plot import plot_plotly, plot_components_plotly

## Tuning Hyperparameters for Prophet

We make a copy of rs and set df as a copy of rs. This is so we don't mix up our variables. We will tune hyperparameters using df in the code below.

In [66]:
rs_saved = rs.copy()
df = rs.copy()

In [67]:
## Adding in holidays

date_range = pd.date_range(start="2020-01-01", end="2026-02-28")

# Generate US federal holidays
calendar = USFederalHolidayCalendar()
holidays = calendar.holidays(start=date_range.min(), end=date_range.max())

# Build the DataFrame in the same structure as your original
federal_holidays = pd.DataFrame({
    'holiday': 'federal_us',
    'ds': pd.to_datetime(holidays),
    'lower_window': 0,
    'upper_window': 1,
})

holidays = federal_holidays

In [68]:
## To tune for hyperparameters, add more possible parameters to the dictionary below and add more values to it.
## So far, the I've been able to get is {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 5}

init_days = '2054 days'
cv_period = '14 days'
forecast_horizon = '14 days'

param_grid = {  
    'changepoint_prior_scale': [0.1, 1],
    'seasonality_prior_scale': [0.1, 1, 5, 10],
}

# Generate all combinations of parameters
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]
rmses = []  # Store the RMSEs for each params here
performance = []

# Use cross validation to evaluate all parameters
for params in all_params:
    params['holidays'] = holidays
    m = Prophet(**params).fit(df)  # Fit model with given params
    df_cv = cross_validation(m, initial = init_days, period=cv_period, horizon = forecast_horizon)
    df_p = performance_metrics(df_cv, rolling_window=14)
    performance.append(df_p)
    rmses.append(df_p['rmse'].values[0])

# Find the best parameters
tuning_results = pd.DataFrame(all_params)
tuning_results['rmse'] = rmses

best_params = all_params[np.argmin(rmses)]

01:37:49 - cmdstanpy - INFO - Chain [1] start processing
01:37:49 - cmdstanpy - INFO - Chain [1] done processing


  0%|          | 0/14 [00:00<?, ?it/s]

01:37:50 - cmdstanpy - INFO - Chain [1] start processing
01:37:50 - cmdstanpy - INFO - Chain [1] done processing
01:37:50 - cmdstanpy - INFO - Chain [1] start processing
01:37:50 - cmdstanpy - INFO - Chain [1] done processing
01:37:50 - cmdstanpy - INFO - Chain [1] start processing
01:37:50 - cmdstanpy - INFO - Chain [1] done processing
01:37:50 - cmdstanpy - INFO - Chain [1] start processing
01:37:51 - cmdstanpy - INFO - Chain [1] done processing
01:37:51 - cmdstanpy - INFO - Chain [1] start processing
01:37:51 - cmdstanpy - INFO - Chain [1] done processing
01:37:51 - cmdstanpy - INFO - Chain [1] start processing
01:37:51 - cmdstanpy - INFO - Chain [1] done processing
01:37:51 - cmdstanpy - INFO - Chain [1] start processing
01:37:52 - cmdstanpy - INFO - Chain [1] done processing
01:37:52 - cmdstanpy - INFO - Chain [1] start processing
01:37:52 - cmdstanpy - INFO - Chain [1] done processing
01:37:52 - cmdstanpy - INFO - Chain [1] start processing
01:37:52 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/14 [00:00<?, ?it/s]

01:37:54 - cmdstanpy - INFO - Chain [1] start processing
01:37:54 - cmdstanpy - INFO - Chain [1] done processing
01:37:54 - cmdstanpy - INFO - Chain [1] start processing
01:37:55 - cmdstanpy - INFO - Chain [1] done processing
01:37:55 - cmdstanpy - INFO - Chain [1] start processing
01:37:55 - cmdstanpy - INFO - Chain [1] done processing
01:37:55 - cmdstanpy - INFO - Chain [1] start processing
01:37:55 - cmdstanpy - INFO - Chain [1] done processing
01:37:55 - cmdstanpy - INFO - Chain [1] start processing
01:37:55 - cmdstanpy - INFO - Chain [1] done processing
01:37:56 - cmdstanpy - INFO - Chain [1] start processing
01:37:56 - cmdstanpy - INFO - Chain [1] done processing
01:37:56 - cmdstanpy - INFO - Chain [1] start processing
01:37:56 - cmdstanpy - INFO - Chain [1] done processing
01:37:56 - cmdstanpy - INFO - Chain [1] start processing
01:37:56 - cmdstanpy - INFO - Chain [1] done processing
01:37:57 - cmdstanpy - INFO - Chain [1] start processing
01:37:57 - cmdstanpy - INFO - Chain [1]

...

01:38:17 - cmdstanpy - INFO - Chain [1] start processing
01:38:17 - cmdstanpy - INFO - Chain [1] done processing
01:38:18 - cmdstanpy - INFO - Chain [1] start processing
01:38:18 - cmdstanpy - INFO - Chain [1] done processing
01:38:18 - cmdstanpy - INFO - Chain [1] start processing
01:38:19 - cmdstanpy - INFO - Chain [1] done processing
01:38:19 - cmdstanpy - INFO - Chain [1] start processing
01:38:19 - cmdstanpy - INFO - Chain [1] done processing
01:38:19 - cmdstanpy - INFO - Chain [1] start processing
01:38:20 - cmdstanpy - INFO - Chain [1] done processing
01:38:20 - cmdstanpy - INFO - Chain [1] start processing
01:38:20 - cmdstanpy - INFO - Chain [1] done processing
01:38:21 - cmdstanpy - INFO - Chain [1] start processing
01:38:21 - cmdstanpy - INFO - Chain [1] done processing
01:38:21 - cmdstanpy - INFO - Chain [1] start processing
01:38:22 - cmdstanpy - INFO - Chain [1] done processing
01:38:22 - cmdstanpy - INFO - Chain [1] start processing
01:38:22 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/14 [00:00<?, ?it/s]

01:38:26 - cmdstanpy - INFO - Chain [1] start processing
01:38:26 - cmdstanpy - INFO - Chain [1] done processing
01:38:27 - cmdstanpy - INFO - Chain [1] start processing
01:38:27 - cmdstanpy - INFO - Chain [1] done processing
01:38:27 - cmdstanpy - INFO - Chain [1] start processing
01:38:28 - cmdstanpy - INFO - Chain [1] done processing
01:38:28 - cmdstanpy - INFO - Chain [1] start processing
01:38:28 - cmdstanpy - INFO - Chain [1] done processing
01:38:28 - cmdstanpy - INFO - Chain [1] start processing
01:38:29 - cmdstanpy - INFO - Chain [1] done processing
01:38:29 - cmdstanpy - INFO - Chain [1] start processing
01:38:30 - cmdstanpy - INFO - Chain [1] done processing
01:38:30 - cmdstanpy - INFO - Chain [1] start processing
01:38:30 - cmdstanpy - INFO - Chain [1] done processing
01:38:30 - cmdstanpy - INFO - Chain [1] start processing
01:38:31 - cmdstanpy - INFO - Chain [1] done processing
01:38:31 - cmdstanpy - INFO - Chain [1] start processing
01:38:31 - cmdstanpy - INFO - Chain [1]

  0%|          | 0/14 [00:00<?, ?it/s]

01:38:35 - cmdstanpy - INFO - Chain [1] start processing
01:38:35 - cmdstanpy - INFO - Chain [1] done processing
01:38:36 - cmdstanpy - INFO - Chain [1] start processing
01:38:36 - cmdstanpy - INFO - Chain [1] done processing
01:38:36 - cmdstanpy - INFO - Chain [1] start processing
01:38:37 - cmdstanpy - INFO - Chain [1] done processing
01:38:37 - cmdstanpy - INFO - Chain [1] start processing
01:38:37 - cmdstanpy - INFO - Chain [1] done processing
01:38:38 - cmdstanpy - INFO - Chain [1] start processing
01:38:38 - cmdstanpy - INFO - Chain [1] done processing
01:38:38 - cmdstanpy - INFO - Chain [1] start processing
01:38:39 - cmdstanpy - INFO - Chain [1] done processing
01:38:39 - cmdstanpy - INFO - Chain [1] start processing
01:38:39 - cmdstanpy - INFO - Chain [1] done processing
01:38:39 - cmdstanpy - INFO - Chain [1] start processing
01:38:40 - cmdstanpy - INFO - Chain [1] done processing
01:38:40 - cmdstanpy - INFO - Chain [1] start processing
01:38:41 - cmdstanpy - INFO - Chain [1]

In [69]:
new_performance = pd.concat(performance, ignore_index=True)

# Round numeric columns for readability
numeric_cols = ["mse", "rmse", "mae", "mape", "mdape", "smape", "coverage"]
new_performance[numeric_cols] = new_performance[numeric_cols].round(4)


new_performance

,horizon,mse,rmse,mae,mape,mdape,smape,coverage
0,14 days,141.4874,11.8948,9.5923,0.3242,0.2445,0.3057,0.8571
1,14 days,142.0671,11.9192,9.6026,0.3243,0.2438,0.3063,0.8827
2,14 days,141.6986,11.9037,9.5887,0.3239,0.2427,0.3054,0.8673
3,14 days,142.3568,11.9313,9.6149,0.3246,0.2439,0.3066,0.8724
4,14 days,145.7551,12.0729,9.6341,0.3262,0.2385,0.3369,0.8622
5,14 days,146.3328,12.0968,9.6615,0.3271,0.2378,0.3394,0.8469
6,14 days,145.7108,12.0711,9.6411,0.3264,0.2376,0.3386,0.8622
7,14 days,146.1150,12.0878,9.6571,0.3269,0.2380,0.3393,0.8571


In [70]:
best_params

{'changepoint_prior_scale': 0.1,
 'seasonality_prior_scale': 0.1,
 'holidays':        holiday         ds  lower_window  upper_window
 0   federal_us 2020-01-01             0             1
 1   federal_us 2020-01-20             0             1
 2   federal_us 2020-02-17             0             1
 3   federal_us 2020-05-25             0             1
 4   federal_us 2020-07-03             0             1
 ..         ...        ...           ...           ...
 63  federal_us 2025-11-27             0             1
 64  federal_us 2025-12-25             0             1
 65  federal_us 2026-01-01             0             1
 66  federal_us 2026-01-19             0             1
 67  federal_us 2026-02-16             0             1
 
 [68 rows x 4 columns]}

In [71]:
m = Prophet(**best_params)
m.add_country_holidays(country_name='US')
m.fit(df)
future = m.make_future_dataframe(periods=14)
forecast = m.predict(future)

01:38:44 - cmdstanpy - INFO - Chain [1] start processing
01:38:44 - cmdstanpy - INFO - Chain [1] done processing


In [72]:
fig1 = plot_plotly(m, forecast)
fig1.show()

fig2 = plot_components_plotly(m, forecast)
fig2.show()